# NUTDTS 816 Time Series Analysis
## L12 The ETS framework

Lab notebook for Chapter 6 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### Carried forward from Lab 11 (run these cells first; they define the objects used below)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing
import tsdata
oilp = tsdata.oil()      # annual oil production, Saudi Arabia, millions of tonnes, 1965-2013 (fpp2 'oil')
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(oilp.index, oilp.values, lw=1, marker='o', ms=3, color='#1B5E3A', label='observed')
fut = pd.date_range(oilp.index[-1], periods=6, freq='YS')[1:]
for a, col in [(0.2, '#B8860B'), (0.6, '#2F6DB5'), (0.9, '#A0302A')]:
    f = SimpleExpSmoothing(oilp, initialization_method='estimated').fit(smoothing_level=a, optimized=False)
    ax.plot(oilp.index, f.fittedvalues.values, lw=1.2, color=col, label=f'SES fitted, α = {a}')
    ax.plot(fut, [f.forecast(5).iloc[0]] * 5, color=col, lw=2)
opt = SimpleExpSmoothing(oilp, initialization_method='estimated').fit()
ax.set_title(f'SES on annual oil production with three α values (optimised α = {opt.params["smoothing_level"]:.2f}); flat forecasts to the right'); ax.legend(fontsize=8); ax.set_xlabel('')
print(pd.DataFrame({'weight on x_T-k (α = 0.2)': [0.2 * 0.8**k for k in range(6)], 'weight on x_T-k (α = 0.6)': [0.6 * 0.4**k for k in range(6)]}, index=[f'k={k}' for k in range(6)]).round(3).to_string())
_caption = 'Small α gives a smooth, slowly adapting level; large α tracks the data closely. All SES forecasts are flat at the final level.'

In [ ]:
air = tsdata.airpassengers(); ann = air.resample('YS').sum() / 1000     # annual passengers, millions, 1949-1960 (trend, no seasonality)
livestock = ann  # short annual series with a clear trend
fig, ax = plt.subplots(figsize=(9, 3.4)); ax.plot(livestock.index, livestock.values, marker='o', ms=3, lw=1, color='#1B5E3A', label='observed')
h = 8; fut = pd.date_range(livestock.index[-1], periods=h + 1, freq='YS')[1:]
for name, kw, col in [('Holt linear', {}, '#B8860B'), ('Holt damped (φ estimated)', {'damped_trend': True}, '#2F6DB5'), ('SES', None, '#555555')]:
    m = (SimpleExpSmoothing(livestock, initialization_method='estimated').fit() if kw is None else Holt(livestock, initialization_method='estimated', **kw).fit())
    ax.plot(fut, m.forecast(h).values, lw=2, color=col, label=name + (f" (φ = {m.params['damping_trend']:.2f})" if kw and kw.get('damped_trend') else ''))
ax.set_title('Annual airline passengers (millions): SES, Holt and damped Holt forecasts'); ax.legend(fontsize=8); ax.set_xlabel('')
_caption = 'SES is flat; Holt extrapolates the recent slope; the damped trend bends toward a limit. Which is right depends on how far ahead and how much you believe the trend persists.'

In [ ]:
grid = tsdata.nigeria_grid(); gtr, gte = grid[:'2025-06'], grid['2025-07':]; h = len(gte)
hw_add = ExponentialSmoothing(gtr, trend='add', seasonal='add', seasonal_periods=12, damped_trend=True, initialization_method='estimated').fit()
hw_mul = ExponentialSmoothing(gtr, trend='add', seasonal='mul', seasonal_periods=12, damped_trend=True, initialization_method='estimated').fit()
fig, ax = plt.subplots(figsize=(9, 3.4)); grid['2022':].plot(ax=ax, lw=1, label='observed')
hw_add.forecast(h).plot(ax=ax, lw=2, color='#B8860B', label='Holt-Winters additive, damped'); hw_mul.forecast(h).plot(ax=ax, lw=1.5, ls='--', color='#2F6DB5', label='Holt-Winters multiplicative, damped')
ax.axvline(gte.index[0], color='#555555', lw=0.8); ax.legend(fontsize=8); ax.set_xlabel(''); ax.set_title('Grid generation (simulated): Holt-Winters forecasts from June 2025')
print(pd.DataFrame({'additive': hw_add.params, 'multiplicative': hw_mul.params}).loc[['smoothing_level', 'smoothing_trend', 'smoothing_seasonal', 'damping_trend']].round(3).to_string())
for name, m in [('HW additive damped', hw_add), ('HW multiplicative damped', hw_mul)]:
    f_ = m.forecast(h); print(f'{name:26s} MAE = {(f_ - gte).abs().mean():5.0f} MW  RMSE = {np.sqrt(((f_ - gte)**2).mean()):5.0f} MW')
_caption = 'On an additive series the two variants give similar forecasts. The estimated parameters show a moderately adaptive level, a fixed slope and a fixed seasonal pattern (β* and γ at zero).'

## The ETS framework

### 6.7 Model selection and forecasting with ETS

In [ ]:
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
def fit_ets(y, error, trend, seasonal, damped, m=12):
    return ETSModel(y, error=error, trend=trend, seasonal=seasonal, damped_trend=damped, seasonal_periods=m, initialization_method='estimated').fit(disp=False)
ap = tsdata.airpassengers(); atr, ate = ap[:'1958-12'], ap['1959-01':]
cands = [('add', None, None, False), ('add', 'add', None, False), ('add', 'add', None, True), ('add', 'add', 'add', True), ('mul', 'add', 'mul', False), ('mul', 'add', 'mul', True), ('mul', None, 'mul', False)]
rows = []
for e, t, s_, d in cands:
    f = fit_ets(atr, e, t, s_, d)
    name = f"ETS({'A' if e=='add' else 'M'},{('N' if t is None else 'A') + ('d' if d else '')},{'N' if s_ is None else ('A' if s_=='add' else 'M')})"
    rows.append({'model': name, 'AICc': round(f.aicc, 1), 'alpha': round(f.params[0], 3)})
print(pd.DataFrame(rows).sort_values('AICc').to_string(index=False))

In [ ]:
best = fit_ets(atr, 'mul', 'add', 'mul', False)
h = len(ate)
sim = best.simulate(nsimulations=h, repetitions=2000, anchor='end', random_state=0)    # simulated future paths
lo80, hi80 = sim.quantile(0.1, axis=1), sim.quantile(0.9, axis=1)
lo95, hi95 = sim.quantile(0.025, axis=1), sim.quantile(0.975, axis=1)
fc = best.forecast(h)
fig, ax = plt.subplots(figsize=(9, 3.4)); ap['1955':].plot(ax=ax, lw=1, label='observed (incl. hold-out)')
fc.plot(ax=ax, lw=2, color='#B8860B', label='ETS(M,A,M) forecast')
ax.fill_between(fc.index, lo95, hi95, color='#B8860B', alpha=0.15, label='95% PI (simulated)'); ax.fill_between(fc.index, lo80, hi80, color='#B8860B', alpha=0.3, label='80% PI')
ax.axvline(ate.index[0], color='#555555', lw=0.8); ax.legend(fontsize=8, ncol=2); ax.set_xlabel(''); ax.set_title('ETS(M,A,M) two-year forecast of airline passengers with simulated intervals')
print(f'Hold-out MAE = {(fc - ate).abs().mean():.1f}   RMSE = {np.sqrt(((fc - ate)**2).mean()):.1f}   (thousand passengers; airline SARIMA in Chapter 5 gave 39.5 / 43.2)')
print(f'80% interval coverage on hold-out: {((ate >= lo80) & (ate <= hi80)).mean():.2f}   95%: {((ate >= lo95) & (ate <= hi95)).mean():.2f}')
_caption = 'Multiplicative-error intervals widen with the level as well as the horizon. The ETS point forecasts run a little below the 1959-60 outcomes, so the 80% interval covers only half the hold-out months while the 95% interval covers nearly all.'

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, AutoARIMA, SeasonalNaive
df = pd.DataFrame({'unique_id': 'air', 'ds': atr.index, 'y': atr.values})
sf = StatsForecast(models=[AutoETS(season_length=12), AutoARIMA(season_length=12), SeasonalNaive(season_length=12)], freq='MS').fit(df)
print('statsforecast AutoETS chose:', sf.fitted_[0, 0].model_['method'])
pred = sf.predict(h=h).set_index('ds')
for col in ['AutoETS', 'AutoARIMA', 'SeasonalNaive']:
    print(f'{col:14s} hold-out MAE = {(pred[col].values - ate.values).__abs__().mean():6.1f}')

## Exercises

5. Fit the full set of seasonal ETS candidates to the `a10` series (hold out 24 months), select by AICc, produce simulated 80% and 95% intervals, and compute their empirical coverage on the hold-out. Compare the point-forecast MAE with the best SARIMA from Chapter 5, Exercise 2.
6. Explain to a warehouse manager, without equations, why their Holt-Winters forecast "keeps the seasonal pattern but flattens the growth after a year", and what the three smoothing parameters in the system mean.
7. The ETS(A,N,N) prediction interval has variance $\sigma^2[1 + (h-1)\alpha^2]$. Compute the ratio of the 12-step to the 1-step interval width for $\alpha = 0.2$ and $\alpha = 0.9$, and interpret.

In [ ]:
# Your work here
